# LIBERO Simulation Benchmark Evaluation

Evaluates a trained `TransformerFlowMatchingPolicy` on the LIBERO benchmark.

**Metric**: Task success rate (%) — fraction of rollouts where the robot completes the task within `max_steps`.

**Standard protocol**: 50 rollouts × 10 tasks per suite, reported per suite and as overall average.

**Suites**: `libero_spatial`, `libero_object`, `libero_goal`, `libero_long`

## 1. Install Dependencies

In [ ]:
# 1. First, install a compatible version of numpy for Python 3.12
!pip install "numpy<2.0"

# 2. Clone LIBERO
import os
if not os.path.exists('LIBERO'):
    !git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git

# 3. Install LIBERO.
# We use --no-deps to prevent it from trying to downgrade numpy to the broken 1.22.4 version.
# Then we manually install the remaining necessary dependencies.
!pip install -e /content/LIBERO --no-deps
!pip install hydra-core==1.2.0

print("LIBERO installed. Please restart the session if imports still fail.")

In [ ]:
# 1. Ensure all core dependencies are present
!pip install bddl robosuite==1.4.1 mujoco

# 2. Advanced compatibility monkeypatching for robosuite 1.4.1
import robosuite
import robosuite.controllers as controllers
import robosuite.robots as robots
from robosuite.robots.robot import Robot
import json
import os

# Fix 1: Manually define load_controller_config for robosuite.controllers
# This is a robust fallback for when the utility module is inaccessible
def load_controller_config(custom_fpath=None, default_controller=None):
    if custom_fpath is not None:
        with open(custom_fpath, 'r') as f:
            return json.load(f)
    # Logic to load default if custom is not provided
    if default_controller is not None:
        # Simplified fallback for monkeypatch
        return {"type": default_controller}
    return {}

if not hasattr(controllers, "load_controller_config"):
    setattr(controllers, "load_controller_config", load_controller_config)
    print("Successfully applied manual patch for load_controller_config")

# Fix 2: Alias SingleArm base class for LIBERO's environment loader
if not hasattr(robots, "SingleArm"):
    setattr(robots, "SingleArm", Robot)

# 3. Final Imports
try:
    import libero
    from libero.libero import benchmark
    from libero.libero.envs import OffScreenRenderEnv
    print("LIBERO, robosuite, and BDDL imported successfully with all compatibility fixes!")
except ImportError as e:
    print(f"Import failed: {e}. If you just downgraded NumPy, you MUST restart the session (Runtime -> Restart session) and then run this cell.")

In [ ]:
# System deps for headless rendering
!apt-get install -y xvfb libgl1-mesa-glx libegl1-mesa libgles2-mesa-dev 2>/dev/null

# Python deps
!pip install -q pyvirtualdisplay


!pip install lerobot==0.4.0
!pip install decord
!pip install opencv-python
!pip install num2words
!pip install transformers


## 2. Setup Virtual Display (Colab Headless Rendering)

In [1]:
import os
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

# Tell MuJoCo to use EGL (GPU-accelerated headless rendering on Colab)
os.environ["MUJOCO_GL"] = "egl"
os.environ["DISPLAY"] = ":99"

print("Virtual display started.")

Virtual display started.


## 3. Clone Repo and Setup Path

In [2]:
import sys
from pathlib import Path

# Clone the repo if not already present
if not Path("lerobot-piper").exists():
    !git clone https://github.com/mayuanyang/lerobot-piper.git

repo_src = Path("lerobot-piper/src")
if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

print(f"Python path includes: {repo_src}")

Python path includes: lerobot-piper/src


## 4. Imports

In [ ]:
import os
import numpy as np

# Fix: Ensure NumPy is < 2.0 to satisfy Numba/robosuite requirements
try:
    import numpy
    if int(numpy.__version__.split('.')[0]) >= 2:
        print(f"Detected NumPy {numpy.__version__}. Downgrading for compatibility...")
        !pip install "numpy<2.0"
        print("Please restart the session (Runtime -> Restart session) and run this cell again.")
except ImportError:
    pass

import json
import torch
from collections import deque
from pathlib import Path
import pandas as pd
import huggingface_hub

from models.transformer_flow_matching.transformer_flow_matching_policy import TransformerFlowMatchingPolicy

# Note: These imports depend on the monkeypatch in cell fGFbrDO2yeX4
from libero.libero import benchmark
from libero.libero.envs import OffScreenRenderEnv

# Device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
print(f"Using NumPy version: {np.__version__}")

## 5. Configuration

In [4]:
# ── Checkpoint ────────────────────────────────────────────────────────────────
# HuggingFace repo ID  OR  local path to a checkpoint directory
CHECKPOINT = "ISdept/fm64-libero"   # <-- change this

# ── Evaluation settings ───────────────────────────────────────────────────────
# Suites to evaluate. Comment out any you want to skip.
SUITES = [
    "libero_spatial",
    "libero_object",
    "libero_goal",
    "libero_long",
]

NUM_ROLLOUTS = 50    # rollouts per task (paper standard = 50)
MAX_STEPS    = 300   # max env steps per rollout
IMAGE_SIZE   = 256   # must match training resolution

# ── Action chunking ───────────────────────────────────────────────────────────
# How many predicted actions to execute before re-planning.
# Smaller = more reactive but slower; larger = faster but less reactive.
# Set to 1 to re-plan every step (most conservative).
N_ACTION_STEPS_TO_EXECUTE = 8

print(f"Checkpoint : {CHECKPOINT}")
print(f"Suites     : {SUITES}")
print(f"Rollouts   : {NUM_ROLLOUTS} per task")
print(f"Max steps  : {MAX_STEPS}")
print(f"Action chunk: {N_ACTION_STEPS_TO_EXECUTE} steps before re-plan")

Checkpoint : ISdept/fm64-libero
Suites     : ['libero_spatial', 'libero_object', 'libero_goal', 'libero_long']
Rollouts   : 50 per task
Max steps  : 300
Action chunk: 8 steps before re-plan


## 6. Load Policy

In [ ]:
import json
import torch
import huggingface_hub
from pathlib import Path
import sys
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata


# Ensure the local repository is in the path for policy classes
repo_src = Path("lerobot-piper/src")
if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

from models.transformer_flow_matching.transformer_flow_matching_policy import TransformerFlowMatchingPolicy
from models.transformer_flow_matching.processor_transformer_flow_matching import make_pre_post_processors

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = Path(CHECKPOINT)
if not ckpt_path.exists():
    print(f"Downloading checkpoint from HuggingFace: {CHECKPOINT}")
    ckpt_path = Path(huggingface_hub.snapshot_download(CHECKPOINT))

print(f"Loading policy from: {ckpt_path}")
policy = TransformerFlowMatchingPolicy.from_pretrained(ckpt_path)
policy.eval()
policy.to(device)

dataset_id = 'lerobot/libero'
# Load dataset metadata and stats using the LeRobot utility as suggested
try:
    print(f"Fetching metadata for {dataset_id}...")
    dataset_metadata = LeRobotDatasetMetadata(dataset_id, force_cache_sync=True, revision="main")
    stats = dataset_metadata.stats

    preprocessor, postprocessor = make_pre_post_processors(policy.config, dataset_stats=stats)
    if isinstance(preprocessor, torch.nn.Module):
        preprocessor.to(device)
    print("Pre/Post processors initialized using LeRobotDatasetMetadata stats.")
except Exception as e:
    print(f"Could not load metadata from hub: {e}. Falling back to local file.")
    stats_path = ckpt_path / "dataset_stats.json"
    if stats_path.exists():
        with open(stats_path, "r") as f:
            stats = json.load(f)
        preprocessor, postprocessor = make_pre_post_processors(policy.config, dataset_stats=stats)
        if isinstance(preprocessor, torch.nn.Module):
            preprocessor.to(device)
        print("Pre/Post processors initialized from local dataset_stats.json.")
    else:
        print("Warning: No stats found. Using identity processors.")
        preprocessor, postprocessor = lambda x: x, lambda x: x

n_obs_steps = policy.config.n_obs_steps
horizon      = policy.config.horizon
action_dim   = policy.config.action_dim
state_dim    = policy.config.state_dim

# Precompute action unnormalization tensors from dataset stats.
# predict_action_chunk returns MEAN_STD normalized values; we must invert before
# sending to the LIBERO environment.
if stats and 'action' in stats:
    action_mean = torch.tensor(stats['action']['mean'], dtype=torch.float32).to(device)
    action_std  = torch.tensor(stats['action']['std'],  dtype=torch.float32).to(device)
    print(f"\n  Action unnorm  mean={action_mean.tolist()}")
    print(f"                 std ={action_std.tolist()}")
else:
    action_mean = None
    action_std  = None
    print("WARNING: no action stats found — actions will NOT be unnormalized.")

# Build the LIBERO-env-key → policy-camera-key mapping.
# The policy only looks at cameras_for_vision_state_concat; any key not in that
# list is silently ignored, so a mismatch here makes the policy completely blind.
POLICY_CAM_KEYS = policy.config.cameras_for_vision_state_concat
# LIBERO provides exactly two cameras in this fixed order:
LIBERO_CAM_NAMES = ["agentview_image", "robot0_eye_in_hand_image"]
# Zip in order — assumes the checkpoint was trained on LIBERO with the same camera order.
cam_key_map = {
    libero_k: policy_k
    for libero_k, policy_k in zip(LIBERO_CAM_NAMES, POLICY_CAM_KEYS)
}

print(f"\nPolicy loaded successfully.")
print(f"  n_obs_steps : {n_obs_steps}")
print(f"  horizon     : {horizon}")
print(f"  action_dim  : {action_dim}")
print(f"  state_dim   : {state_dim}")
print(f"  cameras_for_vision_state_concat: {POLICY_CAM_KEYS}")
print(f"  cam_key_map : {cam_key_map}")

## 7. Inspect Environment Observation Keys (Debug)

Run this once to verify which LIBERO observation keys map to your policy inputs.

In [ ]:
benchmark_dict = benchmark.get_benchmark_dict()
task_suite     = benchmark_dict["libero_goal"]()
task           = task_suite.get_task(0)
bddl_file      = task_suite.get_task_bddl_file_path(0)

env = OffScreenRenderEnv(**{
    "bddl_file_name": bddl_file,
    "camera_heights": IMAGE_SIZE,
    "camera_widths":  IMAGE_SIZE,
})
obs = env.reset()
env.close()

print("LIBERO env observation keys and shapes:")
for k, v in obs.items():
    v = np.array(v)
    print(f"  {k:40s}  shape={v.shape}  dtype={v.dtype}")

print(f"\nPolicy expects:")
print(f"  observation.state              shape=({n_obs_steps}, {state_dim})")
print(f"  observation.images.image       shape=(3, {IMAGE_SIZE}, {IMAGE_SIZE})")
print(f"  observation.images.image2      shape=(3, {IMAGE_SIZE}, {IMAGE_SIZE})")
print(f"  action                         shape=({action_dim},)")

## 8. Observation Preprocessing Utilities

Maps LIBERO env obs → policy input batch.

**Verify the `build_state` mapping against the debug output above.**
The default assumes `robot0_joint_pos` (7-dim) + one gripper scalar = 8-dim.
Adjust if your `state_dim` or obs keys differ.

In [ ]:
def img_to_tensor(img: np.ndarray) -> torch.Tensor:
    """(H, W, C) uint8 → (C, H, W) float32 in [0, 1]."""
    t = torch.from_numpy(img.copy()).float() / 255.0
    return t.permute(2, 0, 1)  # CHW


def build_state(obs: dict) -> np.ndarray:
    """
    Build the proprioceptive state vector from a LIBERO observation dict.

    lerobot/libero observation.state (8-dim) verified from dataset stats:
      indices 0–5: robot0_joint_pos[:6]  — 6 arm joints with large std (~0.1–0.9 rad)
      indices 6–7: robot0_gripper_qpos   — 2 gripper fingers, symmetric, std ~0.014

    NOTE: robot0_joint_pos has 7 values (Panda is 7-DOF), but the lerobot/libero
    dataset only stores the first 6 in observation.state alongside both gripper
    fingers. Using joint_pos[:7] + gripper[:1] gives the wrong alignment.
    """
    joint_pos = np.array(obs["robot0_joint_pos"])     # (7,)  — take first 6
    gripper   = np.array(obs["robot0_gripper_qpos"])  # (2,)  — take both fingers
    return np.concatenate([joint_pos[:6], gripper])    # (8,)


def obs_to_frame(obs: dict) -> dict:
    """
    Convert a raw LIBERO obs dict to a storable frame.

    Images are stored under the policy camera keys (from cam_key_map) so that
    build_batch can pass them directly to the policy without any re-keying.
    Mismatch here makes the policy completely blind (zero vision tokens).
    """
    frame = {"state": build_state(obs)}
    for libero_key, policy_key in cam_key_map.items():
        frame[policy_key] = img_to_tensor(obs[libero_key])
    return frame


def build_batch(obs_buffer: deque, task_description: str) -> dict:
    """
    Convert a deque of recent observations into a batch dict for the policy.

    State: stacked across n_obs_steps → (1, n_obs_steps, state_dim).
    Images: current (last) frame only, under the exact policy camera keys.
    """
    # State: (1, n_obs_steps, state_dim)
    states = torch.stack(
        [torch.tensor(o["state"], dtype=torch.float32) for o in obs_buffer]
    ).unsqueeze(0).to(device)

    batch = {
        "observation.state": states,
        "task_description":  [task_description],
    }

    # Images: one frame per camera under the exact keys the policy expects
    for policy_key in cam_key_map.values():
        batch[policy_key] = obs_buffer[-1][policy_key].unsqueeze(0).to(device)

    return batch


print("Preprocessing utilities defined.")
print(f"Camera mapping: {cam_key_map}")

# Sanity check: print expected state dim
print(f"Expected state_dim={state_dim}, build_state produces 8 (6 joints + 2 gripper)")

In [ ]:
# ── Verify build_state matches the dataset's observation.state ─────────────────
# The policy was trained on lerobot/libero where observation.state (8-dim) was
# computed by the LeRobot converter. We must reconstruct the SAME vector from
# the LIBERO env obs at inference time or the state input will be misaligned.
#
# Run this cell once before eval to confirm the mapping is correct.

from lerobot.datasets.lerobot_dataset import LeRobotDataset

ds = LeRobotDataset('lerobot/libero', revision='main')
dataset_state = ds[0]['observation.state'].numpy()  # (8,) — what training saw

# Reconstruct from LIBERO env obs using our build_state function
benchmark_dict = benchmark.get_benchmark_dict()
suite_check = benchmark_dict["libero_spatial"]()
task_check  = suite_check.get_task(0)
env_check   = OffScreenRenderEnv(**{
    "bddl_file_name":  suite_check.get_task_bddl_file_path(0),
    "camera_heights":  IMAGE_SIZE,
    "camera_widths":   IMAGE_SIZE,
})
env_obs = env_check.reset()
env_check.close()

env_state = build_state(env_obs)

print("Dataset observation.state (episode 0, frame 0):")
print(" ", dataset_state.tolist())
print()
print("build_state from LIBERO env obs:")
print(" ", env_state.tolist())
print()
print("robot0_joint_pos  (7):", np.array(env_obs["robot0_joint_pos"]).tolist())
print("robot0_gripper_qpos (2):", np.array(env_obs["robot0_gripper_qpos"]).tolist())
print()

# The two vectors won't be identical (different episode/task), but their
# STRUCTURE must match — joint dims must be in the same positions.
# Check: are the last two dims of the dataset state small in magnitude (gripper range)?
print("Dataset state last-2 dims:", dataset_state[-2:])
print("  → if these are small (~0.01–0.04), they are gripper fingers.")
print("  → if large (~1–3 rad), they are arm joints and build_state is wrong.")

## 9. Single Episode Evaluation

In [ ]:
def run_episode(env, task_description: str, seed: int) -> bool:
    """
    Run one rollout. Returns True if the task succeeds.

    Action chunking: the policy predicts `horizon` actions at once.
    We execute N_ACTION_STEPS_TO_EXECUTE of them before re-planning.
    """
    obs = env.reset()
    env.env.seed(seed)

    # Observation buffer — padded with the first obs until full.
    # obs_to_frame stores images under the policy camera keys so build_batch
    # passes them directly without any re-keying mismatch.
    first_frame = obs_to_frame(obs)
    obs_buffer = deque(
        [first_frame] * n_obs_steps,
        maxlen=n_obs_steps,
    )

    action_queue = deque()   # pre-computed actions waiting to be executed

    for step in range(MAX_STEPS):
        # Re-plan when the action queue is empty
        if not action_queue:
            batch = build_batch(obs_buffer, task_description)
            batch = preprocessor(batch)

            with torch.no_grad():
                # Returns (1, horizon, action_dim) in MEAN_STD normalized space
                actions = policy.predict_action_chunk(batch)

            # Unnormalize: invert the MEAN_STD applied during training.
            # Without this step the env receives (value - mean)/std ≈ [-3, 3]
            # instead of real joint targets → robot moves erratically or not at all.
            if action_mean is not None:
                actions = actions * action_std + action_mean

            # Queue up the first N_ACTION_STEPS_TO_EXECUTE actions
            chunk = actions[0, :N_ACTION_STEPS_TO_EXECUTE].cpu().numpy()
            action_queue.extend(chunk)

        action = action_queue.popleft()   # (action_dim,)

        obs, reward, done, info = env.step(action)

        # Update observation buffer with new frame
        obs_buffer.append(obs_to_frame(obs))

        if done:
            return True

    return False  # timed out


print("Episode evaluation function defined.")

## 10. Suite Evaluation

In [23]:
def evaluate_suite(suite_name: str) -> dict:
    """
    Evaluate all tasks in a LIBERO suite.
    Returns a dict with per-task and overall success rates.
    """
    print(f"\n{'='*60}")
    print(f"Suite: {suite_name}")
    print(f"{'='*60}")

    benchmark_dict = benchmark.get_benchmark_dict()
    if suite_name not in benchmark_dict:
        raise ValueError(f"Unknown suite '{suite_name}'. "
                         f"Available: {list(benchmark_dict.keys())}")

    task_suite  = benchmark_dict[suite_name]()
    num_tasks   = task_suite.get_num_tasks()
    results     = {}

    print('The num of tasks', num_tasks)

    for task_id in range(num_tasks):
        task        = task_suite.get_task(task_id)
        task_name   = task.name
        bddl_file   = task_suite.get_task_bddl_file_path(task_id)
        task_desc   = task.language



        env = OffScreenRenderEnv(**{
            "bddl_file_name": bddl_file,
            "camera_heights": IMAGE_SIZE,
            "camera_widths":  IMAGE_SIZE,
        })

        print(f"  Task {task_id:2d}/{num_tasks-1}  {task_name}")

        successes = 0
        for rollout_idx in range(NUM_ROLLOUTS):
            success = run_episode(env, task_desc, seed=rollout_idx)
            successes += int(success)
            print(f"  Task {task_id:2d}/{num_tasks-1}  rollout {rollout_idx+1:3d}/{NUM_ROLLOUTS}  "
                  f"success={'✓' if success else '✗'}  "
                  f"running={successes/(rollout_idx+1)*100:.1f}%",
                  end="\r")

        env.close()

        rate = successes / NUM_ROLLOUTS * 100
        results[task_name] = {"successes": successes, "total": NUM_ROLLOUTS, "rate": rate}
        print(f"  Task {task_id:2d} [{task_name}]  → {rate:.1f}%  ({successes}/{NUM_ROLLOUTS})")

    suite_rate = np.mean([v["rate"] for v in results.values()])
    print(f"\n  {suite_name} average: {suite_rate:.1f}%")
    return {"tasks": results, "suite_avg": suite_rate}


print("Suite evaluation function defined.")

Suite evaluation function defined.


## 11. Run Evaluation

In [ ]:
all_results = {}

for suite in SUITES:
    try:
        all_results[suite] = evaluate_suite(suite)
    except Exception as e:
        print(f"\nSkipped {suite}: {e}")

print("\n\nEvaluation complete.")

## 12. Results

In [ ]:
# Per-suite summary table
rows = []
for suite, data in all_results.items():
    for task_name, task_data in data["tasks"].items():
        rows.append({
            "Suite":    suite,
            "Task":     task_name,
            "Success": f"{task_data['successes']}/{task_data['total']}",
            "Rate (%)": f"{task_data['rate']:.1f}",
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Overall summary
print("\n" + "="*50)
print("Suite Averages:")
suite_avgs = []
for suite, data in all_results.items():
    avg = data["suite_avg"]
    suite_avgs.append(avg)
    print(f"  {suite:20s}  {avg:5.1f}%")

if suite_avgs:
    overall = np.mean(suite_avgs)
    print(f"  {'Overall avg':20s}  {overall:5.1f}%")